# On-Sky MCMC Tutorial

This is the same idea as the global MCMC fit for the internal calibrations. I won't add much comments since the process is the same, except you use your on-sky CSV and data.

In [1]:
# IMPORTING DATA

from pyPolCal.csv_tools import read_csv_physical_model_all_bins
from importlib.resources import files
from pathlib import Path

# Defining Path to my CSVs
csvdir = Path('/home/thomasmc/pyPolCal/pyPolCal/CHARIS/datacsvs/onsky_nbs/HD293396')

# Reading in data
interleaved_values_all, interleaved_stds_all, configuration_list_all = read_csv_physical_model_all_bins(csvdir, m3=True)

/home/thomasmc/miniconda3/envs/charisenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# GENERATE MODEL
from pyPolCal.utils import generate_system_mueller_matrix

system_dict = {
    "components" : {
        "wollaston" : {
        "type" : "fitted_wollaston_function_12_28_2025",
        "properties" : {"beam": 'o'}, 
        "tag": "internal",
        },

        "nbs_rot": {
            "type": "rotator_function",
            "properties": {"pa": 90},
            "tag": "internal",
        },
        "image_rotator" : {
        "type" : "fitted_derotator_function_12_28_2025",
        "properties" : {"delta_theta":1.384e-02}, 
        "tag": "internal",
        },
        
        "hwp" : {
            "type" : "two_layer_HWP_function", # Joost 't Hart 2021 HWP model
            "properties" : {"w_SiO2":1.638, "w_MgF2":1.28, "delta_theta": -3.168e-02},
            "tag": "internal",
        },

        "altitude_rot" : {
            "type" : "rotator_function",
            "properties" : {"pa":0},
            "tag":"internal",
        },
        "M3" : {
            "type" : "SUBARU_M3_function",
            "properties" : {"delta_theta":0},
            "tag": "internal",
        },

        "parang_rot" : {
            "type" : "rotator_function",
            "properties" : {"pa":0},
            "tag":"internal",
        },
        },
}
system_mm = generate_system_mueller_matrix(system_dict)
system_mm.evaluate()

array([[ 0.48902582, -0.48857441,  0.00881718,  0.01906714],
       [ 0.48902582, -0.48857441,  0.00881718,  0.01906714],
       [ 0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        ,  0.        ,  0.        ]])

In [2]:

# Define starting guesses

m1, b1, m2, b2 = (1.781,12.47,2.264,14.67) # from minimize
p0_dict = {
    "M3":{"m1": m1,
    "b1": b1,
    "m2": m2,
    "b2": b2
    }
}
m1_bounds = (0.5*m1, 2*m1)
m2_bounds = (0.5*m2, 2*m2)
b1_bounds = (0.5*b1, 2*b1)
b2_bounds = (0.5*b2, 2*b2)
boundslist = [m1_bounds, b1_bounds, m2_bounds, b2_bounds]

bounds_dict = {
    
    "M3" : {
        "m1": m1_bounds,
        "b1": b1_bounds,
        "m2": m2_bounds,
        "b2": b2_bounds
    }
}

# Define priors
prior_dict = {

    "M3": {
        "m1": {"type": "uniform", "kwargs": {"low":0.5*m1, "high": 2*m1}},
        "b1": {"type": "uniform", "kwargs": {"low":0.5*b1, "high": 2*b1}},
        "m2": {"type": "uniform", "kwargs": {"low":0.5*m2, "high": 2*m2}},
        "b2": {"type": "uniform", "kwargs": {"low":0.5*b2, "high": 2*b2}},
}}


In [4]:
from pyPolCal.instruments_jax import run_mcmc, process_model, process_dataset 

# Path for the h5 emcee output file
output_h5 = 'mcmc_tutorial_output_onsky.h5'

ndim = 6  # Number of parameters to fit
pool_processes = 12 # Number of CPU cores to use
nwalkers = max(2 * ndim, pool_processes * 2) # Number of walkers at least twice the number of dimensions
if nwalkers % pool_processes != 0:
    nwalkers += pool_processes - (nwalkers % pool_processes)

print(f"{nwalkers} walkers for {ndim} parameters")
sampler, p_keys = run_mcmc(p0_dict, system_mm, interleaved_values_all,configuration_list_all,prior_dict,bounds_dict,output_h5,nwalkers=nwalkers,pool_processes=pool_processes,process_model=process_model, process_dataset=process_dataset,nsteps=7000, include_sums=False)

24 walkers for 6 parameters



  0%|                                                 | 0/7000 [00:00<?, ?it/s]


  0%|                                       | 1/7000 [00:02<4:01:34,  2.07s/it]


  0%|                                       | 2/7000 [00:04<4:01:24,  2.07s/it]


  0%|                                       | 3/7000 [00:06<4:00:14,  2.06s/it]

...


100%|███████████████████████████████████▉| 6997/7000 [4:28:13<00:11,  3.74s/it]


100%|███████████████████████████████████▉| 6998/7000 [4:28:17<00:07,  3.71s/it]


100%|███████████████████████████████████▉| 6999/7000 [4:28:20<00:03,  3.66s/it]


100%|████████████████████████████████████| 7000/7000 [4:28:24<00:00,  3.77s/it]


100%|████████████████████████████████████| 7000/7000 [4:28:24<00:00,  2.30s/it]

(the excessive outputs are from me running this notebook remotely on screen, it's not like this if you just run the notebook normally)

In [ ]:
# check results
from pyPolCal.plotting import summarize_median_posterior
output_h5 = 'mcmc_tutorial_output_onsky.h5'
summarize_median_posterior(output_h5,p0_dict,step_range=(5000,6000))

Posterior Medians and 1 sigma Credible Intervals:
M3,m1: 2.25567 (+0.94801/-1.00093)
M3,b1: 16.65557 (+5.91659/-7.35054)
M3,m2: 2.88121 (+1.19119/-1.31144)
M3,b2: 20.23299 (+6.62005/-8.76463)


{'M3': {'m1': {'median': np.float64(2.2556679820506105),
   '-1sigma': np.float64(1.0009311535723444),
   '+1sigma': np.float64(0.9480055842033348)},
  'b1': {'median': np.float64(16.65556946997875),
   '-1sigma': np.float64(7.350540609592885),
   '+1sigma': np.float64(5.916585392653801)},
  'm2': {'median': np.float64(2.8812137820857173),
   '-1sigma': np.float64(1.311440082307187),
   '+1sigma': np.float64(1.1911939296266474)},
  'b2': {'median': np.float64(20.232990421554874),
   '-1sigma': np.float64(8.764632663415656),
   '+1sigma': np.float64(6.620051329190069)}}}